# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, leveraging the [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n\nPublished: {metadata.datePublished}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Each entity is referenced by its unique `@id`, as required for working with this dataset and the mlcroissant API.

In [ ]:
# List all record sets and their @ids
print("Available Record Sets in the Package:")
for record_set in dataset.record_sets:
    print(f"- @id: {record_set.id} | name: {getattr(record_set, 'name', 'N/A')}")

# For each record set, list its fields and columns with their @id
for record_set in dataset.record_sets:
    print(f"\nRecord Set: {getattr(record_set, 'name', 'N/A')} (@id: {record_set.id})")
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - Field: {getattr(field, 'name', 'N/A')} (@id: {field.id})")
            # If this field extracts from columns, show columns too
            if hasattr(field, 'columns'):
                print("      Columns:")
                for column in field.columns:
                    print(f"        - {column.id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All selections by `@id`.

In [ ]:
# Identify all record set @ids for extraction
record_set_ids = [record_set.id for record_set in dataset.record_sets]

# Create a dictionary to hold DataFrames by record set @id
dataframes = {}
for record_set_id in record_set_ids:
    # Load records from each record set by its @id
    records = list(dataset.records(record_set=record_set_id))
    if records:  # skip if empty
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(records)} records for record set "{record_set_id}"')
    else:
        print(f'No records found for record set "{record_set_id}"')

# If at least one record set had records, preview its columns
if len(dataframes) > 0:
    # Pick the first non-empty record set
    first_rs_id = next(iter(dataframes))
    print(f'\nColumns for record set {first_rs_id}:')
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print('No records to display.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by attributes. All column selections use the column or field `@id`.

In [ ]:
# Identify the record set and numeric field for EDA
if len(dataframes) == 0:
    print("No data available for EDA.")
else:
    # Select first available record set and try to pick a likely numeric field
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    
    # Try to guess a numeric field by datatype or by known field @id (e.g. using column name heuristics)
    # If unsure, print columns for manual selection
    print(f"Columns: {df.columns.tolist()}")
    
    # For this example, let's use the first column with type float or int if possible
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric fields detected for EDA.")
    else:
        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalize the values (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a group field (categorical)
        possible_group_fields = [col for col in df.columns if df[col].dtype == "O"]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data (mean {numeric_field}) by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical group field available for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below is an example histogram and box plot for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field is not None:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    
    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field])
    plt.title(f'Box plot of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we leveraged the Croissant schema and the `mlcroissant` library to explore the FAIR^2 dataset package, loading its schema, record sets, and performing initial EDA. Each entity was referenced strictly by its `@id` for reproducibility and clarity, as recommended for FAIR datasets.

Key findings and next steps will depend on the specific dataset fields, but the workflow here provides a robust basis for deeper clinical/biomarker analyses and model development.